# Marker GPU Benchmark

Benchmark autocontido de extracao de PDF com Marker. Roda **direto na(s) GPU(s) do notebook**.

**Target:** PDF juridico digitalizado (OCR pesado)

---

## Workflow

1. **Celula 1** - Setup: detecta GPU(s), VRAM, imports
2. **Celula 2** - Config: PDF, chunk size, parametros
3. **Celula 3** - Pilot: 10 paginas em 1 GPU para calibrar ETA
4. **Celula 4** - Benchmark: multi-GPU paralelo ou single-GPU sequencial
5. **Celula 5** - Dashboard visual (graficos + score)
6. **Celula 6** - Export para Volume

---

**Para trocar GPU:** Sidebar -> Compute -> GPU -> selecionar tipo e **quantidade**

**Multi-GPU:** Selecione ex. 4x T4. A Celula 4 distribui chunks entre GPUs automaticamente.

**Custom image:** `marker_benchmark_env` (deploy: `modal deploy marker_benchmark_image.py`)

**Volume:** `marker-benchmark-data` montado em `/mnt/marker-benchmark-data/`

In [ ]:
# ==============================================================================
# CELULA 1: SETUP E DIAGNOSTICO
# ==============================================================================
# Detecta GPU(s), VRAM, pacotes. Nenhuma config necessaria.
# ==============================================================================

import gc
import json
import os
import time
import warnings

warnings.filterwarnings("ignore")

import matplotlib
import pandas as pd
import torch

matplotlib.use("Agg")
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.notebook import tqdm

# --- Hardware ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "N/A"
VRAM_TOTAL_GB = torch.cuda.mem_get_info()[1] / 1e9 if DEVICE == "cuda" else 0
VRAM_FREE_GB = torch.cuda.mem_get_info()[0] / 1e9 if DEVICE == "cuda" else 0
NUM_GPUS = torch.cuda.device_count() if DEVICE == "cuda" else 0


# --- Funcoes de utilidade ---


def section_header(title):
    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}")


def fmt_time(seconds):
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        m, s = divmod(seconds, 60)
        return f"{int(m)}m{int(s)}s"
    else:
        h, rem = divmod(seconds, 3600)
        m, s = divmod(rem, 60)
        return f"{int(h)}h{int(m)}m{int(s)}s"


def fmt_cost(usd):
    return f"${usd:.4f}"


def vram_status(device_id=0):
    if DEVICE != "cuda":
        return {"used_gb": 0, "total_gb": 0, "free_gb": 0, "pct": 0}
    free, total = torch.cuda.mem_get_info(device_id)
    used = total - free
    return {
        "used_gb": round(used / 1e9, 2),
        "total_gb": round(total / 1e9, 2),
        "free_gb": round(free / 1e9, 2),
        "pct": round(used / total * 100, 1),
    }


def clear_vram():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def get_vram_peak_mb(device_id=0):
    """VRAM real do driver NVIDIA."""
    if DEVICE != "cuda":
        return 0
    free, total = torch.cuda.mem_get_info(device_id)
    return round((total - free) / 1e6)


# --- Diagnostico ---
section_header("DIAGNOSTICO DO AMBIENTE")

diag_rows = []
diag_rows.append(
    {"Item": "Device", "Valor": DEVICE, "Status": "OK" if DEVICE == "cuda" else "SEM GPU"}
)
diag_rows.append(
    {"Item": "Num GPUs", "Valor": str(NUM_GPUS), "Status": "OK" if NUM_GPUS > 0 else "--"}
)
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    free, total = torch.cuda.mem_get_info(i)
    diag_rows.append(
        {
            "Item": f"GPU {i}",
            "Valor": f"{name} | {total / 1e9:.1f} GB total | {free / 1e9:.1f} GB livre",
            "Status": "OK",
        }
    )
diag_rows.append({"Item": "PyTorch", "Valor": torch.__version__, "Status": "OK"})
diag_rows.append(
    {
        "Item": "CUDA",
        "Valor": torch.version.cuda or "N/A",
        "Status": "OK" if torch.version.cuda else "--",
    }
)
diag_rows.append(
    {
        "Item": "Volume",
        "Valor": "/mnt/marker-benchmark-data",
        "Status": "OK" if os.path.exists("/mnt/marker-benchmark-data") else "NAO MONTADO",
    }
)

display(pd.DataFrame(diag_rows).style.set_caption("Diagnostico"))

# Verificar Marker
try:
    import marker

    print(f"Marker: OK (versao {getattr(marker, '__version__', '?')})")
except ImportError:
    print("Marker: NAO INSTALADO -- verifique a custom image")

print(f"\nPronto. {NUM_GPUS}x {GPU_NAME} ({VRAM_TOTAL_GB:.0f} GB cada)")
if NUM_GPUS > 1:
    print(f"Multi-GPU detectado! Celula 4 vai distribuir chunks entre {NUM_GPUS} GPUs.")

In [ ]:
# ==============================================================================
# CELULA 2: CONFIGURACAO
# ==============================================================================
# EDITE AQUI: path do PDF, chunk size, parametros Marker.
# ==============================================================================


# --- PDF ALVO ---
PDF_PATH = "/mnt/marker-benchmark-data/input/payload_teste2005.pdf"


# --- MARKER CONFIG ---
MARKER_CONFIG = {
    "output_format": "markdown",
    "paginate_output": True,
    "disable_image_extraction": True,  # Sem imagens = mais rapido
    "force_ocr": False,  # True = forca OCR em tudo (mais lento)
}


# --- CHUNK SIZE ---
# 100 = sweet spot. Amortiza overhead de setup do PdfConverter.
# CHUNK_SIZE=5 gera 200 rebuilds para 1000 pags e trava (testado).
# Para T4 (16GB): 100 funciona bem. Se der OOM, tente 50.
CHUNK_SIZE = 100


# --- PILOT ---
PILOT_PAGES = 10  # Paginas para calibrar ETA


# --- PRECOS GPU (USD/hora, referencia Modal fev 2026) ---
GPU_PRICES = {
    "NVIDIA T4": 0.59,
    "NVIDIA L4": 0.80,
    "NVIDIA A10G": 1.10,
    "NVIDIA A100-SXM4-40GB": 3.50,
    "NVIDIA A100-SXM4-80GB": 3.50,
    "NVIDIA H100": 4.25,
}


def get_gpu_price():
    for key in GPU_PRICES:
        if key in GPU_NAME:
            return GPU_PRICES[key]
    for key in ["T4", "L4", "A10G", "A100", "H100"]:
        if key in GPU_NAME:
            return GPU_PRICES.get(f"NVIDIA {key}", 1.0)
    return 1.0


GPU_PRICE_PER_HOUR = get_gpu_price()


# --- VALIDACAO ---
section_header("CONFIGURACAO")

TOTAL_PAGES = 0
PDF_OK = False

if os.path.exists(PDF_PATH):
    pdf_size_mb = os.path.getsize(PDF_PATH) / 1e6
    print(f"PDF: {PDF_PATH}")
    print(f"Tamanho: {pdf_size_mb:.1f} MB")

    import pypdfium2 as pdfium

    doc = pdfium.PdfDocument(PDF_PATH)
    TOTAL_PAGES = len(doc)
    doc.close()
    print(f"Paginas: {TOTAL_PAGES}")
    PDF_OK = True
else:
    print(f"PDF NAO ENCONTRADO: {PDF_PATH}")
    print("Upload via sidebar Files ou:")
    print("  modal volume put marker-benchmark-data ./seu.pdf /input/payload_teste2005.pdf")

n_chunks = (TOTAL_PAGES + CHUNK_SIZE - 1) // CHUNK_SIZE if TOTAL_PAGES > 0 else 0

config_data = {
    "Parametro": [
        "PDF",
        "Paginas",
        "Tamanho",
        "GPU",
        "VRAM/GPU",
        "Preco/hora/GPU",
        "Chunk size",
        "Chunks",
        "Pilot pages",
        "Force OCR",
    ],
    "Valor": [
        os.path.basename(PDF_PATH),
        str(TOTAL_PAGES),
        f"{pdf_size_mb:.1f} MB" if PDF_OK else "N/A",
        f"{NUM_GPUS}x {GPU_NAME}",
        f"{VRAM_TOTAL_GB:.0f} GB",
        fmt_cost(GPU_PRICE_PER_HOUR),
        str(CHUNK_SIZE),
        str(n_chunks),
        str(PILOT_PAGES),
        str(MARKER_CONFIG.get("force_ocr", False)),
    ],
}
display(pd.DataFrame(config_data).style.set_caption("Configuracao do Benchmark"))

if NUM_GPUS > 1:
    chunks_per_gpu = (n_chunks + NUM_GPUS - 1) // NUM_GPUS
    print(
        f"\nMulti-GPU: {n_chunks} chunks distribuidos entre {NUM_GPUS} GPUs (~{chunks_per_gpu} chunks/GPU)"
    )
    print(f"Custo total/hora: {fmt_cost(GPU_PRICE_PER_HOUR * NUM_GPUS)} ({NUM_GPUS} GPUs)")

In [ ]:
# ==============================================================================
# CELULA 3: PILOT RUN - CALIBRACAO DE ETA
# ==============================================================================
# Processa amostra pequena em 1 GPU para medir velocidade.
# Usa paginas do MEIO do PDF (mais representativo que capa/indice).
# ==============================================================================

if not PDF_OK:
    print("PDF nao carregado. Execute Celula 2.")
else:
    section_header(f"PILOT RUN: {PILOT_PAGES} PAGINAS em {GPU_NAME} (GPU 0)")

    mid = TOTAL_PAGES // 2
    pilot_range = list(range(mid, min(mid + PILOT_PAGES, TOTAL_PAGES)))
    print(f"Paginas: {pilot_range[0] + 1} a {pilot_range[-1] + 1} (meio do PDF)")

    # Carregar modelos Marker (GPU 0 apenas)
    print("\nCarregando modelos Marker...")
    vram_before_load = get_vram_peak_mb()
    t_load_start = time.time()

    from marker.converters.pdf import PdfConverter
    from marker.models import create_model_dict
    from marker.output import text_from_rendered

    model_dict = create_model_dict()
    t_model_load = time.time() - t_load_start
    vram_after_load = get_vram_peak_mb()
    print(f"Modelos carregados em {t_model_load:.1f}s")
    print(f"VRAM pos-load: {vram_after_load} MB (+{vram_after_load - vram_before_load} MB)")

    # Converter pilot
    print(f"\nConvertendo {len(pilot_range)} paginas...")
    config = MARKER_CONFIG.copy()
    config["page_range"] = pilot_range

    t_convert_start = time.time()
    converter = PdfConverter(artifact_dict=model_dict, config=config)
    rendered = converter(PDF_PATH)
    text, _, _ = text_from_rendered(rendered)
    t_convert = time.time() - t_convert_start

    vram_peak = get_vram_peak_mb()

    pilot_pps = len(pilot_range) / t_convert if t_convert > 0 else 0.01
    pilot_cps = len(text) / t_convert if t_convert > 0 else 0

    PILOT_RESULT = {
        "gpu": GPU_NAME,
        "pages": len(pilot_range),
        "model_load_s": round(t_model_load, 2),
        "convert_s": round(t_convert, 2),
        "pages_per_second": round(pilot_pps, 3),
        "chars_per_second": round(pilot_cps, 1),
        "chars": len(text),
        "words": len(text.split()),
        "vram_peak_mb": vram_peak,
        "vram_models_mb": vram_after_load,
    }

    display(
        pd.DataFrame([PILOT_RESULT])
        .T.rename(columns={0: "Valor"})
        .style.set_caption("Resultado do Pilot")
    )

    # ETA
    section_header("ESTIMATIVA PARA BENCHMARK COMPLETO")

    rebuild_overhead_per_chunk = 2.0
    if NUM_GPUS > 1:
        # Multi-GPU: cada GPU processa ~n_chunks/NUM_GPUS chunks
        chunks_per_gpu = (n_chunks + NUM_GPUS - 1) // NUM_GPUS
        eta_per_gpu = (chunks_per_gpu * CHUNK_SIZE) / pilot_pps + rebuild_overhead_per_chunk * (
            chunks_per_gpu - 1
        )
        # Wall time = GPU mais lenta (model load paralelo + conversao)
        ETA_TOTAL = t_model_load + eta_per_gpu
        ETA_COST = (ETA_TOTAL / 3600) * GPU_PRICE_PER_HOUR * NUM_GPUS
        mode_str = f"{NUM_GPUS}x {GPU_NAME} PARALELO"
    else:
        eta_convert = TOTAL_PAGES / pilot_pps
        eta_overhead = rebuild_overhead_per_chunk * (n_chunks - 1)
        ETA_TOTAL = eta_convert + eta_overhead
        ETA_COST = (ETA_TOTAL / 3600) * GPU_PRICE_PER_HOUR
        mode_str = f"1x {GPU_NAME} SEQUENCIAL"

    eta_finish = datetime.now() + timedelta(seconds=ETA_TOTAL)

    eta_data = {
        "Metrica": [
            "Modo",
            "Throughput/GPU (pilot)",
            "ETA total",
            "Custo estimado",
            "Conclusao ~",
        ],
        "Valor": [
            mode_str,
            f"{pilot_pps:.2f} pags/s",
            fmt_time(ETA_TOTAL),
            fmt_cost(ETA_COST),
            eta_finish.strftime("%H:%M"),
        ],
    }
    display(
        pd.DataFrame(eta_data).style.set_caption(
            f"ETA: {TOTAL_PAGES} pags / {n_chunks} chunks de {CHUNK_SIZE}"
        )
    )

    print("\n--- Preview (primeiros 500 chars) ---")
    print(text[:500])

    # Limpar pilot (model_dict usado so no single-GPU path)
    del converter, rendered
    clear_vram()
    if NUM_GPUS == 1:
        print("\nModelos Marker mantidos em VRAM (single-GPU).")
    else:
        # Multi-GPU: cada processo vai carregar seus proprios modelos
        del model_dict
        clear_vram()
        print("\nModelos liberados. Cada GPU carregara seus proprios modelos na Celula 4.")
    print(f"VRAM GPU 0: {vram_status()['used_gb']} GB / {vram_status()['total_gb']} GB")

In [ ]:
# ==============================================================================
# CELULA 4: BENCHMARK COMPLETO
# ==============================================================================
# Se NUM_GPUS == 1: processa sequencialmente (model_dict da Celula 3)
# Se NUM_GPUS > 1: spawna 1 processo por GPU, cada um com CUDA_VISIBLE_DEVICES
#   isolado, modelos proprios, e chunks designados.
#   Wall time = modelo load + chunk mais lento.
# ==============================================================================

import multiprocessing as mp


def _gpu_worker(gpu_id, chunk_list, pdf_path, marker_config, result_queue):
    """
    Worker que roda num processo separado com 1 GPU isolada.
    Carrega modelos, processa chunks, envia resultados via queue.
    """
    import os

    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    import gc
    import time

    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "N/A"

    # Carregar modelos
    t_load = time.time()
    from marker.converters.pdf import PdfConverter
    from marker.models import create_model_dict
    from marker.output import text_from_rendered

    my_models = create_model_dict()
    model_load_s = time.time() - t_load

    # Sinalizar que modelos carregaram
    result_queue.put(
        {"type": "model_loaded", "gpu_id": gpu_id, "model_load_s": round(model_load_s, 2)}
    )

    # Processar chunks
    for chunk_info in chunk_list:
        chunk_id = chunk_info["chunk_id"]
        pages = chunk_info["pages"]

        t_start = time.time()
        try:
            config = marker_config.copy()
            config["page_range"] = pages
            converter = PdfConverter(artifact_dict=my_models, config=config)
            rendered = converter(pdf_path)
            text, _, _ = text_from_rendered(rendered)
            convert_s = time.time() - t_start

            free, total = torch.cuda.mem_get_info(0)
            vram_mb = round((total - free) / 1e6)

            result_queue.put(
                {
                    "type": "chunk_done",
                    "gpu_id": gpu_id,
                    "chunk_id": chunk_id,
                    "pages": len(pages),
                    "page_range": f"{pages[0] + 1}-{pages[-1] + 1}",
                    "convert_s": round(convert_s, 2),
                    "chars": len(text),
                    "words": len(text.split()),
                    "vram_mb": vram_mb,
                    "pps": round(len(pages) / convert_s, 3) if convert_s > 0 else 0,
                    "text": text,
                    "status": "ok",
                }
            )

            del converter, rendered
            gc.collect()
            torch.cuda.empty_cache()

        except Exception as e:
            convert_s = time.time() - t_start
            result_queue.put(
                {
                    "type": "chunk_done",
                    "gpu_id": gpu_id,
                    "chunk_id": chunk_id,
                    "pages": len(pages),
                    "page_range": f"{pages[0] + 1}-{pages[-1] + 1}",
                    "convert_s": round(convert_s, 2),
                    "status": "error",
                    "error": str(e),
                    "text": "",
                }
            )

    result_queue.put({"type": "worker_done", "gpu_id": gpu_id})


# --- Execucao ---
if not PDF_OK:
    print("PDF nao carregado. Execute Celula 2.")
else:
    # Gerar chunks
    all_pages = list(range(TOTAL_PAGES))
    chunks = []
    for i, start in enumerate(range(0, TOTAL_PAGES, CHUNK_SIZE)):
        pages = all_pages[start : start + CHUNK_SIZE]
        chunks.append({"chunk_id": i, "pages": pages})
    n_chunks = len(chunks)

    if NUM_GPUS <= 1:
        # =====================================================
        # SINGLE GPU: sequencial com model_dict da Celula 3
        # =====================================================
        if "model_dict" not in dir():
            print("Modelos nao carregados. Execute Celula 3 primeiro.")
        else:
            section_header(f"BENCHMARK: {TOTAL_PAGES} PAGINAS em 1x {GPU_NAME}")
            print(f"{TOTAL_PAGES} paginas -> {n_chunks} chunks de ~{CHUNK_SIZE}")
            print(f"ETA estimado (pilot): {fmt_time(ETA_TOTAL)}")

            bench_start = time.time()
            CHUNK_RESULTS = []
            all_texts = []

            pbar = tqdm(total=TOTAL_PAGES, desc=f"1x {GPU_NAME}", unit="pag")
            pages_done = 0
            convert_elapsed = 0

            for chunk_info in chunks:
                i = chunk_info["chunk_id"]
                chunk_pages = chunk_info["pages"]
                chunk_start = time.time()

                try:
                    config = MARKER_CONFIG.copy()
                    config["page_range"] = chunk_pages
                    converter = PdfConverter(artifact_dict=model_dict, config=config)
                    rendered = converter(PDF_PATH)
                    text, _, _ = text_from_rendered(rendered)
                    chunk_time = time.time() - chunk_start
                    vram_now = get_vram_peak_mb()

                    CHUNK_RESULTS.append(
                        {
                            "chunk_id": i,
                            "gpu_id": 0,
                            "pages": len(chunk_pages),
                            "page_range": f"{chunk_pages[0] + 1}-{chunk_pages[-1] + 1}",
                            "convert_s": round(chunk_time, 2),
                            "chars": len(text),
                            "words": len(text.split()),
                            "vram_mb": vram_now,
                            "pps": round(len(chunk_pages) / chunk_time, 3) if chunk_time > 0 else 0,
                            "status": "ok",
                        }
                    )
                    all_texts.append(text)

                    pages_done += len(chunk_pages)
                    convert_elapsed += chunk_time
                    pps_real = pages_done / convert_elapsed if convert_elapsed > 0 else 0.01
                    remaining = TOTAL_PAGES - pages_done
                    eta_remaining = remaining / pps_real
                    eta_finish = datetime.now() + timedelta(seconds=eta_remaining)
                    pbar.update(len(chunk_pages))
                    pbar.set_postfix_str(
                        f"{pps_real:.2f} p/s | VRAM {vram_now}MB | "
                        f"ETA {fmt_time(eta_remaining)} (~{eta_finish.strftime('%H:%M')})"
                    )
                    del converter, rendered
                    clear_vram()

                except Exception as e:
                    chunk_time = time.time() - chunk_start
                    print(f"\n  Chunk {i} FALHOU: {e}")
                    CHUNK_RESULTS.append(
                        {
                            "chunk_id": i,
                            "gpu_id": 0,
                            "pages": len(chunk_pages),
                            "page_range": f"{chunk_pages[0] + 1}-{chunk_pages[-1] + 1}",
                            "convert_s": round(chunk_time, 2),
                            "status": "error",
                            "error": str(e),
                        }
                    )
                    all_texts.append("")
                    pbar.update(len(chunk_pages))

            pbar.close()
            bench_wall = time.time() - bench_start

    else:
        # =====================================================
        # MULTI GPU: 1 processo por GPU, chunks distribuidos
        # =====================================================
        section_header(f"BENCHMARK: {TOTAL_PAGES} PAGINAS em {NUM_GPUS}x {GPU_NAME} (PARALELO)")

        # Distribuir chunks round-robin entre GPUs
        gpu_assignments = {i: [] for i in range(NUM_GPUS)}
        for idx, chunk_info in enumerate(chunks):
            gpu_id = idx % NUM_GPUS
            gpu_assignments[gpu_id].append(chunk_info)

        for gpu_id, assigned in gpu_assignments.items():
            total_pags = sum(len(c["pages"]) for c in assigned)
            print(f"  GPU {gpu_id}: {len(assigned)} chunks, {total_pags} paginas")

        # Queue para receber resultados dos workers
        result_queue = mp.Queue()

        # Spawnar workers
        print(f"\nSpawnando {NUM_GPUS} workers...")
        bench_start = time.time()
        workers = []
        for gpu_id in range(NUM_GPUS):
            p = mp.Process(
                target=_gpu_worker,
                args=(gpu_id, gpu_assignments[gpu_id], PDF_PATH, MARKER_CONFIG, result_queue),
            )
            p.start()
            workers.append(p)

        # Coletar resultados com progress bar
        CHUNK_RESULTS = []
        all_texts = [""] * n_chunks  # pre-alocar por chunk_id
        pages_done = 0
        workers_done = 0
        models_loaded = 0

        pbar = tqdm(total=TOTAL_PAGES, desc=f"{NUM_GPUS}x {GPU_NAME}", unit="pag")

        while workers_done < NUM_GPUS:
            msg = result_queue.get()

            if msg["type"] == "model_loaded":
                models_loaded += 1
                print(
                    f"  GPU {msg['gpu_id']}: modelos carregados em {msg['model_load_s']}s ({models_loaded}/{NUM_GPUS})"
                )

            elif msg["type"] == "chunk_done":
                CHUNK_RESULTS.append(msg)
                chunk_pages = msg.get("pages", 0)
                pages_done += chunk_pages
                if "text" in msg:
                    all_texts[msg["chunk_id"]] = msg.get("text", "")

                elapsed = time.time() - bench_start
                pps_real = pages_done / elapsed if elapsed > 0 else 0.01
                remaining = TOTAL_PAGES - pages_done
                eta_remaining = remaining / pps_real if pps_real > 0 else 9999
                eta_finish = datetime.now() + timedelta(seconds=eta_remaining)

                status = "OK" if msg.get("status") == "ok" else "ERRO"
                pbar.update(chunk_pages)
                pbar.set_postfix_str(
                    f"GPU{msg['gpu_id']} c{msg['chunk_id']} {status} | "
                    f"{pps_real:.2f} p/s efetivo | "
                    f"ETA {fmt_time(eta_remaining)} (~{eta_finish.strftime('%H:%M')})"
                )

            elif msg["type"] == "worker_done":
                workers_done += 1
                print(f"  GPU {msg['gpu_id']}: worker finalizado ({workers_done}/{NUM_GPUS})")

        pbar.close()

        # Esperar processos terminarem
        for p in workers:
            p.join(timeout=30)

        bench_wall = time.time() - bench_start

        # Ordenar por chunk_id para consistencia
        CHUNK_RESULTS = sorted(
            [r for r in CHUNK_RESULTS if r["type"] == "chunk_done"], key=lambda x: x["chunk_id"]
        )

    # =====================================================
    # RESULTADOS (comum single e multi)
    # =====================================================
    section_header("RESULTADO")

    ok_chunks = [c for c in CHUNK_RESULTS if c.get("status") == "ok"]
    total_chars = sum(c.get("chars", 0) for c in ok_chunks)
    total_words = sum(c.get("words", 0) for c in ok_chunks)
    total_convert = sum(c.get("convert_s", 0) for c in ok_chunks)
    max_vram = max((c.get("vram_mb", 0) for c in ok_chunks), default=0)
    avg_vram = sum(c.get("vram_mb", 0) for c in ok_chunks) / len(ok_chunks) if ok_chunks else 0
    pps_final = pages_done / bench_wall if bench_wall > 0 else 0
    cost_usd = (bench_wall / 3600) * GPU_PRICE_PER_HOUR * NUM_GPUS

    BENCH_RESULT = {
        "gpu": GPU_NAME,
        "num_gpus": NUM_GPUS,
        "vram_gb": VRAM_TOTAL_GB,
        "total_pages": pages_done,
        "total_chars": total_chars,
        "total_words": total_words,
        "wall_time_s": round(bench_wall, 2),
        "convert_time_s": round(total_convert, 2),
        "pages_per_second": round(pps_final, 3),
        "pages_per_second_per_gpu": round(pps_final / NUM_GPUS, 3),
        "chars_per_second": round(total_chars / bench_wall, 1) if bench_wall > 0 else 0,
        "avg_vram_mb": round(avg_vram),
        "max_vram_mb": round(max_vram),
        "cost_usd": round(cost_usd, 4),
        "cost_per_page_usd": round(cost_usd / pages_done, 6) if pages_done > 0 else 0,
        "chunks_ok": len(ok_chunks),
        "chunks_failed": len(CHUNK_RESULTS) - len(ok_chunks),
        "chunk_size": CHUNK_SIZE,
        "model_load_s": PILOT_RESULT["model_load_s"],
        "price_per_hour": GPU_PRICE_PER_HOUR,
        "price_per_hour_total": GPU_PRICE_PER_HOUR * NUM_GPUS,
    }

    display(
        pd.DataFrame([BENCH_RESULT])
        .T.rename(columns={0: "Valor"})
        .style.set_caption(f"Benchmark: {TOTAL_PAGES} pags em {NUM_GPUS}x {GPU_NAME}")
    )

    # Tabela por chunk
    section_header("DETALHES POR CHUNK")
    chunk_df = pd.DataFrame(CHUNK_RESULTS)
    display_cols = [
        c
        for c in [
            "chunk_id",
            "gpu_id",
            "page_range",
            "pages",
            "convert_s",
            "pps",
            "chars",
            "vram_mb",
            "status",
        ]
        if c in chunk_df.columns
    ]
    display(chunk_df[display_cols].style.set_caption("Chunks"))

    # Per-GPU summary (multi-GPU)
    if NUM_GPUS > 1 and "gpu_id" in chunk_df.columns:
        section_header("RESUMO POR GPU")
        gpu_summary = (
            chunk_df[chunk_df["status"] == "ok"]
            .groupby("gpu_id")
            .agg(
                {
                    "pages": "sum",
                    "convert_s": "sum",
                    "chars": "sum",
                    "vram_mb": "max",
                    "pps": "mean",
                }
            )
            .round(2)
        )
        gpu_summary.columns = [
            "Paginas",
            "Tempo Convert (s)",
            "Chars",
            "VRAM Pico (MB)",
            "PPS Media",
        ]
        display(gpu_summary.style.set_caption("Performance por GPU"))

    print(f"\nBenchmark concluido em {fmt_time(bench_wall)}")
    print(f"Throughput efetivo: {pps_final:.2f} pags/s ({pps_final / NUM_GPUS:.2f} pags/s/GPU)")
    print(f"Custo: {fmt_cost(cost_usd)} ({fmt_cost(cost_usd / pages_done)}/pag)")

In [ ]:
# ==============================================================================
# CELULA 5: DASHBOARD VISUAL
# ==============================================================================
# Graficos de performance por chunk + metricas consolidadas.
# ==============================================================================

if "CHUNK_RESULTS" not in dir() or not CHUNK_RESULTS:
    print("Sem resultados. Execute Celula 4.")
else:
    ok_chunks = [c for c in CHUNK_RESULTS if c.get("status") == "ok"]
    if not ok_chunks:
        print("Todos os chunks falharam.")
    else:
        chunk_df = pd.DataFrame(ok_chunks)
        has_multi_gpu = NUM_GPUS > 1 and "gpu_id" in chunk_df.columns

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(
            f"Marker Benchmark: {TOTAL_PAGES} pags | {NUM_GPUS}x {GPU_NAME} | {VRAM_TOTAL_GB:.0f}GB/GPU",
            fontsize=13,
            fontweight="bold",
        )

        # Cores por GPU
        gpu_colors = [
            "#2196F3",
            "#4CAF50",
            "#FF5722",
            "#FF9800",
            "#9C27B0",
            "#00BCD4",
            "#E91E63",
            "#795548",
        ]

        # 1. Tempo por chunk (colorido por GPU)
        ax = axes[0, 0]
        if has_multi_gpu:
            for gid in sorted(chunk_df["gpu_id"].unique()):
                mask = chunk_df["gpu_id"] == gid
                color = gpu_colors[gid % len(gpu_colors)]
                ax.bar(
                    chunk_df.loc[mask, "chunk_id"],
                    chunk_df.loc[mask, "convert_s"],
                    color=color,
                    alpha=0.8,
                    label=f"GPU {gid}",
                )
            ax.legend(fontsize=8)
        else:
            ax.bar(chunk_df["chunk_id"], chunk_df["convert_s"], color="#2196F3", alpha=0.8)
        ax.axhline(
            chunk_df["convert_s"].mean(),
            color="red",
            linestyle="--",
            label=f"media: {chunk_df['convert_s'].mean():.1f}s",
        )
        ax.set_xlabel("Chunk")
        ax.set_ylabel("Tempo (s)")
        ax.set_title("Tempo por Chunk")

        # 2. Throughput por chunk
        ax = axes[0, 1]
        if has_multi_gpu:
            for gid in sorted(chunk_df["gpu_id"].unique()):
                mask = chunk_df["gpu_id"] == gid
                color = gpu_colors[gid % len(gpu_colors)]
                ax.bar(
                    chunk_df.loc[mask, "chunk_id"],
                    chunk_df.loc[mask, "pps"],
                    color=color,
                    alpha=0.8,
                    label=f"GPU {gid}",
                )
            ax.legend(fontsize=8)
        else:
            ax.bar(chunk_df["chunk_id"], chunk_df["pps"], color="#4CAF50", alpha=0.8)
        ax.axhline(chunk_df["pps"].mean(), color="red", linestyle="--")
        ax.set_xlabel("Chunk")
        ax.set_ylabel("Paginas/s")
        ax.set_title("Throughput por Chunk")

        # 3. VRAM por chunk
        ax = axes[1, 0]
        if has_multi_gpu:
            for gid in sorted(chunk_df["gpu_id"].unique()):
                mask = chunk_df["gpu_id"] == gid
                color = gpu_colors[gid % len(gpu_colors)]
                ax.plot(
                    chunk_df.loc[mask, "chunk_id"],
                    chunk_df.loc[mask, "vram_mb"],
                    marker="o",
                    color=color,
                    label=f"GPU {gid}",
                )
            ax.legend(fontsize=8)
        else:
            ax.plot(chunk_df["chunk_id"], chunk_df["vram_mb"], marker="o", color="#FF5722")
        ax.axhline(
            VRAM_TOTAL_GB * 1000,
            color="gray",
            linestyle=":",
            label=f"VRAM total: {VRAM_TOTAL_GB:.0f} GB",
        )
        ax.set_xlabel("Chunk")
        ax.set_ylabel("VRAM (MB)")
        ax.set_title("VRAM ao Longo do Tempo")

        # 4. Caracteres por chunk
        ax = axes[1, 1]
        if has_multi_gpu:
            for gid in sorted(chunk_df["gpu_id"].unique()):
                mask = chunk_df["gpu_id"] == gid
                color = gpu_colors[gid % len(gpu_colors)]
                ax.bar(
                    chunk_df.loc[mask, "chunk_id"],
                    chunk_df.loc[mask, "chars"],
                    color=color,
                    alpha=0.8,
                    label=f"GPU {gid}",
                )
            ax.legend(fontsize=8)
        else:
            ax.bar(chunk_df["chunk_id"], chunk_df["chars"], color="#9C27B0", alpha=0.8)
        ax.axhline(chunk_df["chars"].mean(), color="red", linestyle="--")
        ax.set_xlabel("Chunk")
        ax.set_ylabel("Caracteres")
        ax.set_title("Texto Extraido por Chunk")

        plt.tight_layout()
        chart_path = "/tmp/marker_benchmark_dashboard.png"
        plt.savefig(chart_path, dpi=150, bbox_inches="tight")
        plt.show()

        # --- Sumario ---
        section_header("SUMARIO")

        summary_data = {
            "Metrica": [
                "GPU Config",
                "VRAM Total",
                "Paginas Processadas",
                "Tempo Total (wall)",
                "Throughput efetivo",
                "Throughput/GPU",
                "Chars/s",
                "VRAM Pico",
                "Custo Total",
                "Custo/Pagina",
                "Chunks OK",
                "Chunks Falhos",
                "Variancia Throughput",
                "Chunk mais rapido",
                "Chunk mais lento",
            ],
            "Valor": [
                f"{NUM_GPUS}x {GPU_NAME}",
                f"{VRAM_TOTAL_GB * NUM_GPUS:.0f} GB ({VRAM_TOTAL_GB:.0f}/GPU)",
                str(BENCH_RESULT["total_pages"]),
                fmt_time(BENCH_RESULT["wall_time_s"]),
                f"{BENCH_RESULT['pages_per_second']:.2f} pags/s",
                f"{BENCH_RESULT['pages_per_second_per_gpu']:.2f} pags/s",
                f"{BENCH_RESULT['chars_per_second']:.0f}",
                f"{BENCH_RESULT['max_vram_mb']} MB ({BENCH_RESULT['max_vram_mb'] / 1000 / VRAM_TOTAL_GB * 100:.0f}% de 1 GPU)",
                fmt_cost(BENCH_RESULT["cost_usd"]),
                fmt_cost(BENCH_RESULT["cost_per_page_usd"]),
                str(BENCH_RESULT["chunks_ok"]),
                str(BENCH_RESULT["chunks_failed"]),
                f"{chunk_df['pps'].std():.3f} pags/s",
                f"Chunk {chunk_df.loc[chunk_df['pps'].idxmax(), 'chunk_id']}: {chunk_df['pps'].max():.2f} p/s",
                f"Chunk {chunk_df.loc[chunk_df['pps'].idxmin(), 'chunk_id']}: {chunk_df['pps'].min():.2f} p/s",
            ],
        }
        display(pd.DataFrame(summary_data).style.set_caption("Sumario do Benchmark"))

In [ ]:
# ==============================================================================
# CELULA 6: EXPORT DE RESULTADOS
# ==============================================================================
# Salva no Volume: JSON, CSV, blueprint, texto extraido, grafico.
# ==============================================================================

if "BENCH_RESULT" not in dir():
    print("Sem resultados. Execute Celula 4.")
else:
    OUTPUT_DIR = "/mnt/marker-benchmark-data/results"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ts = int(time.time())
    date_str = datetime.now().strftime("%Y%m%d_%H%M")
    gpu_tag = GPU_NAME.replace(" ", "_").replace("-", "_")
    config_tag = f"{NUM_GPUS}x{gpu_tag}"

    # 1. JSON completo
    json_path = os.path.join(OUTPUT_DIR, f"bench_{config_tag}_{date_str}.json")
    export_data = {
        "timestamp": ts,
        "date": datetime.now().isoformat(),
        "pdf": os.path.basename(PDF_PATH),
        "total_pages": TOTAL_PAGES,
        "config": {
            "chunk_size": CHUNK_SIZE,
            "marker_config": MARKER_CONFIG,
            "pilot_pages": PILOT_PAGES,
            "num_gpus": NUM_GPUS,
        },
        "pilot": PILOT_RESULT,
        "result": BENCH_RESULT,
        "chunks": [{k: v for k, v in c.items() if k != "text"} for c in CHUNK_RESULTS],
    }
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    print(f"JSON: {json_path}")

    # 2. CSV chunks
    csv_path = os.path.join(OUTPUT_DIR, f"bench_{config_tag}_{date_str}.csv")
    csv_data = [{k: v for k, v in c.items() if k not in ("text", "type")} for c in CHUNK_RESULTS]
    pd.DataFrame(csv_data).to_csv(csv_path, index=False)
    print(f"CSV: {csv_path}")

    # 3. Texto extraido completo
    if "all_texts" in dir() and all_texts:
        txt_path = os.path.join(OUTPUT_DIR, f"text_{config_tag}_{date_str}.md")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("\n\n".join(t for t in all_texts if t))
        txt_size = os.path.getsize(txt_path) / 1e6
        print(f"Texto: {txt_path} ({txt_size:.1f} MB)")

    # 4. Grafico
    chart_src = "/tmp/marker_benchmark_dashboard.png"
    chart_dst = os.path.join(OUTPUT_DIR, f"chart_{config_tag}_{date_str}.png")
    if os.path.exists(chart_src):
        import shutil

        shutil.copy2(chart_src, chart_dst)
        print(f"Grafico: {chart_dst}")

    # 5. Blueprint
    bp_path = os.path.join(OUTPUT_DIR, f"blueprint_{config_tag}_{date_str}.json")
    blueprint = {
        "version": "1.0",
        "type": "marker-gpu-benchmark",
        "gpu": GPU_NAME,
        "num_gpus": NUM_GPUS,
        "vram_gb": VRAM_TOTAL_GB,
        "pages": TOTAL_PAGES,
        "wall_time_s": BENCH_RESULT["wall_time_s"],
        "pages_per_second": BENCH_RESULT["pages_per_second"],
        "pages_per_second_per_gpu": BENCH_RESULT["pages_per_second_per_gpu"],
        "cost_usd": BENCH_RESULT["cost_usd"],
        "cost_per_page_usd": BENCH_RESULT["cost_per_page_usd"],
        "recommendation": {
            "chunk_size": CHUNK_SIZE,
            "config": MARKER_CONFIG,
        },
    }
    with open(bp_path, "w", encoding="utf-8") as f:
        json.dump(blueprint, f, indent=2, ensure_ascii=False)
    print(f"Blueprint: {bp_path}")

    # Sumario
    section_header("EXPORT CONCLUIDO")
    export_files = []
    for p in [json_path, csv_path, bp_path, chart_dst]:
        if os.path.exists(p):
            export_files.append(
                {"Arquivo": os.path.basename(p), "Tamanho": f"{os.path.getsize(p):,} bytes"}
            )
    display(pd.DataFrame(export_files).style.set_caption("Arquivos Exportados"))

    print("\nResultados persistidos no Volume.")
    print(
        f"Para baixar: modal volume get marker-benchmark-data /results/{os.path.basename(json_path)} ."
    )
    print("\nPara comparar configs: rode com N diferente de GPUs e compare os JSONs.")